# RescueAI backend on Google Colab (free, temporary public API)

This spins up the RescueAI FastAPI backend inside this Colab runtime and exposes it publicly via an `ngrok` tunnel, free of charge.

**Good for:** quick testing, sharing a link with a teammate, hitting the API from a phone/Postman.

**Not good for:** a persistent/production deployment - the tunnel URL changes every run, and the runtime disconnects after inactivity or ~12 hours. For an always-on free deployment, use the Hugging Face Space instead (see the main README).

**Before running:** get a free ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken and paste it into Cell 4 below.

## 1. Upload the project
Zip the `rescueai/` folder on your machine, then upload it here (Colab's file browser on the left, or the cell below).

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose your rescueai.zip
!unzip -oq rescueai.zip -d /content/app
%cd /content/app/rescueai/backend
!ls

## 2. Install dependencies

In [ ]:
!pip install -q fastapi uvicorn sqlalchemy pydantic python-dotenv python-multipart pyngrok

## 3. Seed synthetic demo data (resources, hospitals, users)

In [ ]:
!python seed.py

## 4. Run the API and expose it publicly via ngrok

In [ ]:
from pyngrok import ngrok
import subprocess, time

NGROK_AUTHTOKEN = "PASTE_YOUR_FREE_NGROK_AUTHTOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Kill any previous uvicorn/ngrok so re-running this cell is safe
!pkill -f uvicorn || true
ngrok.kill()

proc = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(4)

public_url = ngrok.connect(8000, "http")
print("Public API base:", public_url)
print("Swagger docs:   ", f"{public_url}/docs")
print("Try it: ", f"{public_url}/api/dashboard/stats")

## 5. Try it
Open the Swagger docs URL printed above in a browser, or call the demo endpoint directly:

In [ ]:
import requests
r = requests.post(f"{public_url}/api/demo/load-scenario")
print(r.status_code, r.json())

## 6. Keep it running
Leave this notebook tab open and connected. When you're done, you can stop the tunnel/server with:
```python
ngrok.kill()
proc.terminate()
```